In [ ]:
import json

json_path = '../dataset/original/inference_train.json'
output_path = '../dataset/ellipsis_recovered/train.json'

with open(json_path, 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(len(dataset))

758


In [4]:
def format_dialogue(item):
    """conversation 배열을 '화자 1: 말\n화자 2: 말\n...' 형식의 문자열로 변환"""
    conv = item["input"]["conversation"]
    lines = []
    for turn in conv:
        sp = turn.get("speaker")
        utt = turn.get("utterance", "")
        # "화자 1: "처럼 표현 (스페이스 포함)
        lines.append(f"화자 {sp}: {utt}")
    return "\n".join(lines)


In [2]:
import re, os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import HumanMessage

from dotenv import load_dotenv
load_dotenv()
#os.environ['OPENAI_API_KEY'] = "YOUR_API_KEY"

True

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    request_timeout=60,
    api_key=os.environ["OPENAI_API_KEY"]
)

system_prompt = """
You are an assistant specialized in restoring omitted components in Korean dialogue lines.

**Core Rules (Crucial):**
1) You will receive a dialogue as plain text. Output **only the dialogue**, with **all omitted components minimally restored inside square brackets [ ]**.
2) If there are errors, please correct them. 
3) Preserve the original line breaks, speaker labels, and sentence order exactly. You may correct obvious typos or nonstandard colloquialisms only if it does not alter meaning.
4) Do not change the meaning, tense, or referents (like “that”, “then”, “there”). If you're uncertain, do not over-add — err on the side of minimal restoration.
5) Restoration targets include primarily: subject, object, predicate, auxiliary or connective expressions, adverbials (time/place/reason/manner), particles, demonstratives (like “그거”), etc.

**Processing guidelines (for internal reasoning only):**
- Use discourse context (topic continuity, preceding/following utterances) to identify omitted elements, then restore *only the necessary minimal parts* using [ ].
- For colloquial contractions (e.g., “했지” → “했[지]”, or “해소” → “해서”) restore naturally if meaning remains identical.
- If multiple restorations are needed in one line, place each [ ] at the precise insertion point.

**Output format:**
- Output **only** the dialogue with restorations in square brackets. Nothing else.

---

**Few-shot examples (mimic exactly this format)**

Example 1:
Input:
화자 2: 진짜 신의 한수
화자 1: 이사하자마자 비 많이 와서 베란다 물 많이 새는 거 알았잖아
화자 2: 글치 계속 해떴으면 몰랐겠지
화자 1: 그 때 물새는 거 알고 코킹작업해소 다행이다
화자 2: ㅇㅇ 안그랬으면 오늘처럼 비 많이 내리는 날 물바다됐을거야
화자 1: 요 아래 씽크홀 공사하던데 괜찮을라나
화자 2: 그러게 저번에도 비 많이 와서 땅꺼진 건데 큰일이네
화자 1: 하수도 공사도 같이 하더만 물 안빠져서
화자 2: 새로 지은 곳인데도 그러네
화자 1: 부실공사지 뭐
화자 2: 비 많이 올 때는 그쪽으로 다니지 말아야겠다
화자 1: ㅇㅇ 조심해
화자 1: 저번에 지나가다 보니 좀 무섭더라
화자 2: 나도 봤는데 씽크홀 크기가 엄청나더라
화자 1: 오늘 비가 엄청 많이 내리네

Output:
화자 2: 진짜 [이사한 게] 신의 한수[였어]
화자 1: [우리가] 이사하자마자 비[가] 많이 와서 베란다[에] 물[이] 많이 새는 거 알았잖아
화자 2: 글치 [그걸] 계속 [해가] 떴으면 [우리는] 몰랐겠지
화자 1: 그 때 [베란다에] 물 새는 거 알고 코킹 작업해[서] 다행이다
화자 2: ㅇㅇ 안 그랬으면 오늘처럼 비[가] 많이 내리는 날 [집이] 물바다 됐을 거야
화자 1: 요 아래 씽크홀 공사하던데 [그게] 괜찮을라나
화자 2: 그러게 저번에도 비[가] 많이 와서 땅[이] 꺼진 건데 큰일이네
화자 1: 하수도 공사도 같이 하더만 물[이] 안 빠져서
화자 2: [거기가] 새로 지은 곳인데도 그러네
화자 1: [그건] 부실 공사[지] 뭐
화자 2: 비[가] 많이 올 때는 [우리는] 그쪽으로 다니지 말아야겠다
화자 1: ㅇㅇ 조심해
화자 1: 저번에 [거길] 지나가다 보니 좀 무섭더라
화자 2: 나도 봤는데 [그] 씽크홀[의] 크기[가] 엄청나더라
화자 1: 오늘 비[가] 엄청 많이 내리네

Example 2
Input: 
화자 1: 학교다닐때 어떤 학생이셨나요?
화자 2: 조용하고 튀지 않았어요. name1 님은요!
화자 1: 저도 약간 본성을 숨기고 살았어서 ㅋㅋㅋ 남들이 보면 모범생으로 보이는 그런 학생이었네요
화자 2: 모범생으로 보이려면 학업 성적도 좋고 친구들이랑도 문제없이 지내셨을 거  같네요.
화자 1: 그냥 수업시간에 떠들고 이런거 안하는 학생이었어요 ㅋㅋ 혼나고 이런걸 싫어해서
화자 2: 남녀공학 다니셨나요?
화자 1: 중학교는 여중 고등학교는 공학이요!!
화자 1: name2님은요
화자 2: 저는 다 공학이요.
화자 2: 거의 초등 동창이 고등학교 까지 같이 다녔어요.
화자 1: 아하 동네가 좁았나요?
화자 2: 네네 시골이라서요. 학교가 한 교씩 밖에 없었어요.
화자 1: 아 그럼 거의 모르는 얼굴이 없겠네요
화자 2: 거의 그랬죠. 한 다리 건너면 다 아는 사람이었죠^^
화자 2: 고등학교는 시내 큰 학교로 가고 싶었는데 교통편이 안 좋아서 포기했었어요
화자 1: 아하 그렇게 쭉 같이 다니다 고등학교 졸업하면 엄청 정들거 같은데 어떠셨나요
화자 2: 맨날 보던 애들이라 졸업할 땐 몰랐는데
나중엔 많이 생각나더라구요
화자 2: 지금은 다들 결혼해서 아이있을 나이인데 못본지 오래되서 길에서 봐도 모를거 같아요
화자 1: 아 저도 고향가서 그렇게 스쳐지나가면 가물가물하더라고요.

Output:
화자 1: 학교 다닐 때 어떤 학생이셨나요?
화자 2: 조용하고 튀지 않았어요. name1님은 [어떤 학생이셨나]요?
화자 1: 저도 약간 본성을 숨기며 살았어서, ㅋㅋㅋ 남들이 보면 [제가] 모범생으로 보이는 그런 학생이었네요
화자 2: 모범생으로 보이려면 [아마] 학업 성적도 좋았고, 친구들이랑도 문제없이 지냈을 거 같네요
화자 1: 그냥 수업 시간에 떠들고 이런 거 안 하는 학생이었어요, ㅋㅋ 혼나는 걸 싫어해서요
화자 2: 남녀공학 [학교를] 다니셨나요?
화자 1: 중학교는 여중이었고, 고등학교는 [남녀공학이었어요]!!
화자 1: name2님은요?
화자 2: 저는 [중학교와 고등학교 모두] 공학이었어요
화자 2: 거의 초등학교 동창이 고등학교까지 [같이] 다녔어요
화자 1: 아하, 동네가 좁았나요?
화자 2: 네네, 시골이라서요. 학교가 한 개씩 밖에 없었어요
화자 1: 아, 그럼 거의 모르는 얼굴이 없겠네요
화자 2: 거의 그랬죠. 한 다리만 건너면 다 아는 사람이었죠^^
화자 2: 고등학교는 시내 큰 학교로 가고 싶었는데, 교통편이 안 좋아서 포기했었어요
화자 1: 아하, 그렇게 쭉 같이 다니다가 고등학교 졸업하면 엄청 정들 거 같은데, 어떠셨나요?
화자 2: 맨날 보던 애들이라 졸업할 땐 몰랐었는데, 나중엔 많이 생각나더라구요
화자 2: 지금은 다들 결혼해서 아이 있는 나이인데, 못 본 지 오래되어서 길에서 봐도 모를 것 같아요
화자 1: 아, 저도 고향에 가서 그렇게 스쳐 지나가면 가물가물하더라고요
"""

integrated_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt.strip()),
    ("human", "<Actual Dialouge to Process>\n{actual_dialouge}")
])


In [16]:
from tqdm import tqdm
# 🔢 총 번역 문장 수 계산
total_utterances = sum(len(sample["input"]["conversation"]) for sample in dataset)

# 🔁 번역 처리
translated_dataset = []
progress = tqdm(total=total_utterances, desc="Translating utterances")

for sample in dataset:
    new_sample = sample.copy()
    new_sample["input"] = sample["input"].copy()
    new_sample["input"]["conversation"] = []

    conversation = sample["input"]["conversation"]

    for i in range(len(conversation)):
        start_idx = max(0, i - 2)
        context_slice = conversation[start_idx:i+1]

        if len(context_slice) == 0:
            continue

        # transcript 만들기
        transcript_lines = [
            f"speaker{utt['speaker']}: {utt['utterance']}" for utt in context_slice
        ]
        transcript = "\n".join(transcript_lines)

        # GPT 호출
        try:
            chain = integrated_prompt | llm
            response = chain.invoke({"transcript": transcript})
            translated = response.content.strip()
        except Exception as e:
            translated = f"[Error] {str(e)}"

        # 원 발화 복사 후 번역 추가
        new_utt = conversation[i].copy()
        new_utt["translated_utterance"] = translated
        new_sample["input"]["conversation"].append(new_utt)

        # ✅ tqdm 한 칸 증가
        progress.update(1)

    translated_dataset.append(new_sample)

progress.close()


Translating utterances: 100%|██████████| 3084/3084 [51:10<00:00,  1.00it/s]  


NameError: name 'output_path' is not defined

In [17]:
output_path = '../dataset/dev_english.json'
# 결과 저장
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(translated_dataset, f, ensure_ascii=False, indent=2)

print(f"✅ 저장 완료: {output_path}")

✅ 저장 완료: ../dataset/dev_english.json
